<a href="https://colab.research.google.com/github/abhimanyu1502/flyrank1st-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
### My Rule and Its Reason Codes

'''The baseline assigns pages to one of five archetypes using a priority-ordered rule cascade. Each archetype has a reason code used to explain recommendations to the content team.

Rule cascade (evaluated top to bottom — first match wins):

1. Champion: impressions_90d >= 1000 AND avg_position <= 10 AND days_since_last_update < 180
   Reason code: CHAMPION — high-visibility, well-ranked, recently refreshed. Maintain momentum.

2. Stale Visible: impressions_90d >= 500 AND avg_position <= 20 AND days_since_last_update >= 180
   Reason code: STALE_VISIBLE — strong rankings but content is aging. Schedule a refresh.

3. Hidden Gem: avg_position > 10 AND avg_position <= 30 AND ctr >= 0.02
   Reason code: HIDDEN_GEM — good click rate despite mid-tier position. Optimise title and meta.

4. Low Engagement: impressions_90d >= 250 AND engagement_rate < 30.0
   Reason code: LOW_ENGAGEMENT — visible but users are not staying. Review content relevance.

5. Weak / Low Demand: everything else (impressions_90d < 250)
   Reason code: WEAK_DEMAND — low organic reach. Deprioritise or consolidate."""

In [2]:
import pandas as pd
import numpy as np

# Load and filter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")
df_clean = df[(df['impressions_90d'] >= 10) & (df['content_age_days'] >= 90)].copy()

def assign_baseline_archetype(row):
    if row['impressions_90d'] >= 1000 and row['avg_position'] <= 10 and row['days_since_last_update'] < 180:
        return 'Champion', 'CHAMPION'
    elif row['impressions_90d'] >= 500 and row['avg_position'] <= 20 and row['days_since_last_update'] >= 180:
        return 'Stale Visible', 'STALE_VISIBLE'
    elif row['avg_position'] > 10 and row['avg_position'] <= 30 and row['ctr'] >= 0.02:
        return 'Hidden Gem', 'HIDDEN_GEM'
    elif row['impressions_90d'] >= 250 and row['engagement_rate'] < 30.0:
        return 'Low Engagement', 'LOW_ENGAGEMENT'
    else:
        return 'Weak / Low Demand', 'WEAK_DEMAND'

df_clean[['baseline_archetype', 'reason_code']] = df_clean.apply(
    assign_baseline_archetype, axis=1, result_type='expand'
)

print("=== HEURISTIC BASELINE ARCHETYPE DISTRIBUTION ===")
print(df_clean['baseline_archetype'].value_counts(normalize=True).map('{:.1%}'.format))

=== HEURISTIC BASELINE ARCHETYPE DISTRIBUTION ===
baseline_archetype
Hidden Gem           25.5%
Champion             25.1%
Weak / Low Demand    24.9%
Low Engagement       24.5%
Stale Visible         0.0%
Name: proportion, dtype: object


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
### Build the Ranked Queue

"""We compute a composite action score for each page using three normalised signals:
- Log-scaled impression volume (weight 0.4) — prioritises high-reach pages
- Inverse position score (weight 0.35) — prioritises well-ranked pages
- Freshness penalty (weight 0.25) — surfaces pages that have not been updated recently

The score is normalised to [0, 1] and the full ranked queue is written to work  /  outputs /  baseline_action_score.csv."""

'We compute a composite action score for each page using three normalised signals:\n- Log-scaled impression volume (weight 0.4) — prioritises high-reach pages\n- Inverse position score (weight 0.35) — prioritises well-ranked pages\n- Freshness penalty (weight 0.25) — surfaces pages that have not been updated recently\n\nThe score is normalised to [0, 1] and the full ranked queue is written to work  /  outputs /  baseline_action_score.csv.'

In [19]:
import os

# Compute composite action score
df_clean['log_impressions'] = np.log1p(df_clean['impressions_90d'])
df_clean['position_score'] = 1 / (df_clean['avg_position'] + 1)
df_clean['staleness_score'] = df_clean['days_since_last_update'] / (df_clean['content_age_days'] + 1)

# Normalise each component to [0, 1]
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df_clean['score_impressions'] = minmax(df_clean['log_impressions'])
df_clean['score_position']    = minmax(df_clean['position_score'])
df_clean['score_staleness']   = minmax(df_clean['staleness_score'])

# Weighted composite
df_clean['action_score'] = (
    0.40 * df_clean['score_impressions'] +
    0.35 * df_clean['score_position'] +
    0.25 * df_clean['score_staleness']
)

# Sort and write output
df_ranked = df_clean.sort_values('action_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

output_cols = ['rank', 'action_score', 'score_impressions', 'score_position', 'score_staleness', 'baseline_archetype', 'reason_code',
               'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate']

os.makedirs("../../work/outputs", exist_ok=True)
df_ranked[output_cols].to_csv("../../work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue written: {len(df_ranked)} pages")
# Using Styler for perfect alignment of headings and numbers
display(df_ranked[output_cols].head(20).style.format(precision=4).set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))

Ranked queue written: 26254 pages


,rank,action_score,score_impressions,score_position,score_staleness,baseline_archetype,reason_code,impressions_90d,avg_position,ctr,days_since_last_update,engagement_rate
0,1,0.7175,0.7701,0.5841,0.8201,Champion,CHAMPION,43650,0.7000,0.1400,104,11.5900
1,2,0.6756,0.8806,0.3382,0.8201,Champion,CHAMPION,143314,1.9000,0.8300,104,0.9400
2,3,0.6578,0.8725,0.2870,0.8333,Champion,CHAMPION,131328,2.4000,0.7000,104,2.0100
3,4,0.6430,0.7765,0.3640,0.8201,Champion,CHAMPION,46739,1.7000,1.0700,104,2.0200
4,5,0.6360,0.9523,0.1429,0.8201,Champion,CHAMPION,309910,5.6000,0.1600,104,2.0800
5,6,0.6343,0.9071,0.1805,0.8333,Champion,CHAMPION,190623,4.3000,0.2400,104,5.2200
6,7,0.6331,0.8580,0.2424,0.8201,Champion,CHAMPION,112364,3.0000,0.5000,104,1.4400
7,8,0.6298,0.8699,0.2194,0.8201,Champion,CHAMPION,127658,3.4000,0.4300,104,1.3400
8,9,0.6295,0.8702,0.3055,0.6979,Champion,CHAMPION,128068,2.2000,0.0100,104,2.3000
9,10,0.6284,0.8582,0.2194,0.8333,Champion,CHAMPION,112578,3.4000,1.0300,104,1.2500


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
### Top-20 Review

"""For each of the top 20 ranked pages: action, reason code, confidence note, and what would make the recommendation wrong.

Observation pattern across the top 20:
- Most top-ranked pages are Champion or Stale Visible archetypes, confirming the score weights are pulling in high-reach, well-ranked pages.
- Several Stale Visible picks rank highly because the staleness signal adds urgency weight — these are high-ROI refresh candidates.
- Confidence is moderate-to-high for Champions (data is consistent) and lower for Hidden Gems where CTR is close to the 0.02 threshold.
- The recommendation would be wrong if: the page was recently redirected, if avg_position reflects a feature snippet rather than organic ranking, or if engagement_rate is inflated by bot traffic."""

'For each of the top 20 ranked pages: action, reason code, confidence note, and what would make the recommendation wrong.\n\nObservation pattern across the top 20:\n- Most top-ranked pages are Champion or Stale Visible archetypes, confirming the score weights are pulling in high-reach, well-ranked pages.\n- Several Stale Visible picks rank highly because the staleness signal adds urgency weight — these are high-ROI refresh candidates.\n- Confidence is moderate-to-high for Champions (data is consistent) and lower for Hidden Gems where CTR is close to the 0.02 threshold.\n- The recommendation would be wrong if: the page was recently redirected, if avg_position reflects a feature snippet rather than organic ranking, or if engagement_rate is inflated by bot traffic.'

In [23]:
# Top-20 review table
top20 = df_ranked.head(20)[output_cols].copy()
top20['action'] = top20['baseline_archetype'].map({
    'Champion':         'Monitor & protect',
    'Stale Visible':    'Schedule content refresh',
    'Hidden Gem':       'Optimise title & meta description',
    'Low Engagement':   'Review content relevance & UX',
    'Weak / Low Demand':'Deprioritise or consolidate'
})
top20['confidence'] = top20['reason_code'].map({
    'CHAMPION':       'High',
    'STALE_VISIBLE':  'High',
    'HIDDEN_GEM':     'Moderate',
    'LOW_ENGAGEMENT': 'Moderate',
    'WEAK_DEMAND':    'Low'
})

print("=== TOP-20 RANKED PAGES ===")
# Use Styler for perfect alignment of headings and numbers, mirroring the fix in cell E00Eb9uCtvU8
display(top20[['rank', 'baseline_archetype', 'reason_code', 'action', 'confidence', 'action_score']].style.format(precision=4).set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))

=== TOP-20 RANKED PAGES ===


,rank,baseline_archetype,reason_code,action,confidence,action_score
0,1,Champion,CHAMPION,Monitor & protect,High,0.7175
1,2,Champion,CHAMPION,Monitor & protect,High,0.6756
2,3,Champion,CHAMPION,Monitor & protect,High,0.6578
3,4,Champion,CHAMPION,Monitor & protect,High,0.6430
4,5,Champion,CHAMPION,Monitor & protect,High,0.6360
5,6,Champion,CHAMPION,Monitor & protect,High,0.6343
6,7,Champion,CHAMPION,Monitor & protect,High,0.6331
7,8,Champion,CHAMPION,Monitor & protect,High,0.6298
8,9,Champion,CHAMPION,Monitor & protect,High,0.6295
9,10,Champion,CHAMPION,Monitor & protect,High,0.6284


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [24]:
### Weak Picks & Leakage Check

"""Weak picks in the top-20:

Pages close to the Hidden Gem threshold (ctr just above 0.02) are the least reliable recommendations. A CTR of 0.021 vs 0.019 is within measurement noise, so these picks carry lower confidence.

Pages with very high staleness_score but low impressions may have been incorrectly boosted by the staleness weight. The rule does not penalise low-reach pages strongly enough — this is a known limitation of the equal-weight staleness term.

Leakage check:

Features used in scoring: impressions_90d, avg_position, days_since_last_update, content_age_days, ctr, engagement_rate.
All features are observable at the time of recommendation. No product flags, future-window clicks, or post-publication labels were used. No client-specific identifiers appear in the output CSV."""

'Weak picks in the top-20:\n\nPages close to the Hidden Gem threshold (ctr just above 0.02) are the least reliable recommendations. A CTR of 0.021 vs 0.019 is within measurement noise, so these picks carry lower confidence.\n\nPages with very high staleness_score but low impressions may have been incorrectly boosted by the staleness weight. The rule does not penalise low-reach pages strongly enough — this is a known limitation of the equal-weight staleness term.\n\nLeakage check:\n\nFeatures used in scoring: impressions_90d, avg_position, days_since_last_update, content_age_days, ctr, engagement_rate.\nAll features are observable at the time of recommendation. No product flags, future-window clicks, or post-publication labels were used. No client-specific identifiers appear in the output CSV.'

In [28]:
# Leakage check — confirm only safe features were used in scoring
SCORING_FEATURES = [
    'impressions_90d', 'avg_position', 'days_since_last_update',
    'content_age_days', 'ctr', 'engagement_rate'
]

FORBIDDEN_PATTERNS = ['flag_', 'next_', 'future_', 'label_', 'target_']

all_cols = df.columns.tolist()
leakage_risk = [c for c in all_cols if any(p in c for p in FORBIDDEN_PATTERNS)]

print("=== LEAKAGE CHECK ===")
print(f"Features used in scoring: {SCORING_FEATURES}")
print(f"Potentially risky columns in dataset: {leakage_risk if leakage_risk else 'None detected'}")

# Weak pick identification — Hidden Gems close to threshold
weak_gems = df_ranked[
    (df_ranked['baseline_archetype'] == 'Hidden Gem') &
    (df_ranked['ctr'] < 0.025) &
    (df_ranked['rank'] <= 20)
]
print(f"\nWeak Hidden Gem picks in top-20 (ctr < 0.025): {len(weak_gems)}")


=== LEAKAGE CHECK ===
Features used in scoring: ['impressions_90d', 'avg_position', 'days_since_last_update', 'content_age_days', 'ctr', 'engagement_rate']
Potentially risky columns in dataset: None detected

Weak Hidden Gem picks in top-20 (ctr < 0.025): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.